In [11]:
import pandas as pd
import pyodbc
from sqlalchemy.engine import Engine
from src.utils.db_connector import get_pyodbc_conn_string
from src.utils.db_connector import get_sqlalchemy_engine
from scipy.stats import mode
import numpy as np
ENGINE: Engine = get_sqlalchemy_engine()

In [4]:
TEMP_TABLE_NAME = 'T_TEMP_SILVER_FACT'
print(f"\n--- Loading Silver Master Fact ({TEMP_TABLE_NAME}) from SQL Server ---")

try:
    CONN_STRING = get_pyodbc_conn_string()
    cnxn = pyodbc.connect(CONN_STRING)
    sql_query = f"SELECT * FROM dbo.{TEMP_TABLE_NAME}"

    df_silver_master = pd.read_sql(sql_query, cnxn)
    cnxn.close()
    print(f"✅ Successfully loaded Silver Master Fact. Rows: {len(df_silver_master):,}")
    print(f"Columns: {df_silver_master.shape[1]}")

except Exception as e:
    cnxn.close()
    print(f"❌ FAILED to load Silver Master Fact. Error: {e}")
    print("Action: Ensure the temporary table exists and the DB connection is active.")




--- Loading Silver Master Fact (T_TEMP_SILVER_FACT) from SQL Server ---


C:\Users\Ayush\AppData\Local\Temp\ipykernel_7492\780574505.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_silver_master = pd.read_sql(sql_query, cnxn)


✅ Successfully loaded Silver Master Fact. Rows: 107,261
Columns: 35


In [5]:
DATETIME_COLS_TO_FIX = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in DATETIME_COLS_TO_FIX:
    df_silver_master[col] = pd.to_datetime(df_silver_master[col], errors='coerce')
df_master=df_silver_master.copy()
columns=df_silver_master.columns.tolist()
print(columns)

# Product Tables
print("\n--- Starting Gold Layer: Product Dimension Creation ---")
def product_dimensions(df_master:pd.DataFrame)->pd.DataFrame:
    df_dim_product = df_master.groupby('product_id').agg(
        category_english=('category_english', 'first'),
        product_name_length=('product_name_length', 'median'),
        product_description_length=('product_description_length', 'median'),
        product_photo_qty=('product_photo_qty', 'median'),

        avg_item_price=('item_price', 'mean'),
        total_items_sold=('order_item_id', 'count'),
        avg_review_score=('review_score_stars', 'mean'),
        avg_weight_grams=('product_weight_grams', 'median'),
    ).reset_index()
    return df_dim_product

product_df=product_dimensions(df_master)
try:
    product_df.to_sql(
        name='T_DIM_PRODUCT',
        con=ENGINE,
        schema='gold',
        if_exists='replace',
        index=False
    )
    print("✅ T_DIM_CUSTOMER_HABIT loaded successfully to Gold Schema.")
except Exception as e:
    print(f"❌ FAILED to load Gold Table. Error: {e}")


['order_id', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'product_id', 'seller_id', 'item_price', 'shipping_cost', 'customer_unique_id', 'customer_zip_code_prefix', 'product_name_length', 'product_description_length', 'product_photo_qty', 'product_weight_grams', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_city', 'seller_state', 'category_english', 'review_id', 'review_score_stars', 'review_comment_text', 'review_comment_title', 'review_answer_timestamp', 'total_payment_value', 'max_payment_installments', 'payment_types_count', 'customer_lat', 'customer_lng', 'customer_city', 'customer_state']

--- Starting Gold Layer: Product Dimension Creation ---
✅ T_DIM_CUSTOMER_HABIT loaded successfully to Gold Schema.


In [6]:
# Customer Table
print("\n--- Starting Customer Dimension Aggregation ---")
def customer_dimensions(df_master:pd.DataFrame)->pd.DataFrame:
    df_master['order_purchase_timestamp'] = pd.to_datetime(
        df_master['order_purchase_timestamp'],
        errors='coerce'
    )
    TODAY = pd.to_datetime('today')
    df_master['recency_days'] = (TODAY - df_master['order_purchase_timestamp']).dt.days

    df_dim_customer = df_master.groupby('customer_unique_id').agg(
        customer_city=('customer_city', 'last'),
        customer_state=('customer_state', 'last'),
        customer_lat=('customer_lat', 'last'),
        customer_lng=('customer_lng', 'last'),

        # Aggregated Buying Habits
        recency_days=('recency_days', 'min'),
        total_orders=('order_id', 'nunique'),
        total_items_purchased=('order_item_id', 'count'),
        total_revenue_spent=('item_price', 'sum'),

        # Aggregated Payment/Service Metrics
        max_payment_installments=('max_payment_installments', 'max'),
        avg_payment_types_used=('payment_types_count', 'mean'),

        # Review Score
        avg_customer_review=('review_score_stars', 'mean'),
    ).reset_index()
    return df_dim_customer

customer_df=customer_dimensions(df_master)
customer_df.count()

try:
    customer_df.to_sql(
        name='T_DIM_CUSTOMER_HABIT',
        con=ENGINE,
        schema='gold',
        if_exists='replace',
        index=False
    )
    print("✅ T_DIM_CUSTOMER_HABIT loaded successfully to Gold Schema.")
except Exception as e:
    print(f"❌ FAILED to load Gold Table. Error: {e}")


--- Starting Customer Dimension Aggregation ---
✅ T_DIM_CUSTOMER_HABIT loaded successfully to Gold Schema.


In [7]:
# Sales Fact Tables
print("\n--- Starting Gold Layer: Sales Fact Table Creation ---")

def sales_fact_aggregation(df_master: pd.DataFrame) -> pd.DataFrame:
    COMPOSITE_KEY = ['order_id', 'order_item_id', 'product_id', 'seller_id']

    df_fact_sales = df_master.groupby(COMPOSITE_KEY).agg(
        customer_unique_id=('customer_unique_id', 'first'),
        purchase_timestamp=('order_purchase_timestamp', 'first'),
        delivered_date=('order_delivered_customer_date', 'first'),
        review_answer_timestamp=('review_answer_timestamp', 'first'),

        item_price=('item_price', 'sum'),
        shipping_cost=('shipping_cost', 'sum'),
        total_payment_value=('total_payment_value', 'sum'),
        review_score_stars=('review_score_stars', 'first'),
        max_payment_installments=('max_payment_installments', 'first'),

    ).reset_index()
    df_fact_sales['delivery_delay_days'] = (
        df_fact_sales['delivered_date'] - df_fact_sales['purchase_timestamp']
    ).dt.days
    return df_fact_sales
fact_Sales_df=sales_fact_aggregation(df_master)

try:
    fact_Sales_df.to_sql(
        name='T_Fact_Sales',
        con=ENGINE,
        schema='gold',
        if_exists='replace',
        index=False
    )
    print("✅ T_DIM_CUSTOMER_HABIT loaded successfully to Gold Schema.")
except Exception as e:
    print(f"❌ FAILED to load Gold Table. Error: {e}")


--- Starting Gold Layer: Sales Fact Table Creation ---
✅ T_DIM_CUSTOMER_HABIT loaded successfully to Gold Schema.


In [8]:
def review_fact_aggregation(df_master: pd.DataFrame) -> pd.DataFrame:
    print("\n--- Starting Gold Layer: Review Fact Table Creation ---")

    COMPOSITE_KEY = ['review_id', 'order_id']
    df_fact_review = df_master.groupby(COMPOSITE_KEY).agg(

        # Measures/Content
        review_score_stars=('review_score_stars', 'first'),
        review_comment_text=('review_comment_text', 'first'),
        review_comment_title=('review_comment_title', 'first'),

        # Time Point (When the customer submitted the final feedback)
        review_answer_timestamp=('review_answer_timestamp', 'first'),

        # Link to Seller/Product for context
        seller_id=('seller_id', 'first'),
        product_id=('product_id', 'first'),

    ).reset_index()
    return df_fact_review
fact_review_df=review_fact_aggregation(df_master)

try:
    fact_review_df.to_sql(
        name='T_FACT_REVIEW_SENTIMENT',
        con=ENGINE,
        schema='gold',
        if_exists='replace',
        index=False
    )

    print("✅ T_fact_review loaded successfully to Gold Schema.")
except Exception as e:
    print(f"❌ FAILED to load Gold Table. Error: {e}")


--- Starting Gold Layer: Review Fact Table Creation ---
✅ T_fact_review loaded successfully to Gold Schema.


In [21]:

def create_final_ml_feature_table(df_customer, df_sales, df_product) -> pd.DataFrame:

    print("\n--- Final ML Feature Table Creation ---")

    df_sales_enriched = df_sales.merge(
        df_product[['product_id', 'avg_weight_grams','category_english']],
        on='product_id',
        how='left'
    )
    df_new_features = df_sales_enriched.groupby('customer_unique_id').agg(
        avg_item_weight_g=('avg_weight_grams', 'mean'),
        most_freq_category=('category_english', lambda x: x.mode().iloc[0] if not x.mode().empty else 'unknown')
    ).reset_index()

    df_ml_final = df_customer.merge(
        df_new_features,
        on='customer_unique_id',
        how='left'
    )

    df_ml_final['Y_satisfaction_class'] = np.select(
        [(df_ml_final['avg_customer_review'] >= 4.5), (df_ml_final['avg_customer_review'] >= 3.5)],
        [2, 1],
        default=0
    )

    df_ml_final=df_ml_final.drop(columns='total_orders')
    ENGINE = get_sqlalchemy_engine()
    df_ml_final.to_sql(
        name='T_ML_CUSTOMER_FEATURES', con=ENGINE, schema='gold',
        if_exists='replace', index=False
    )

    print(f"✅ FINAL ML Feature Table loaded to gold.T_ML_CUSTOMER_FEATURES. Rows: {len(df_ml_final):,}")
    return df_ml_final
df_ml_final=create_final_ml_feature_table(customer_df,fact_Sales_df,product_df)






--- Final ML Feature Table Creation ---
✅ FINAL ML Feature Table loaded to gold.T_ML_CUSTOMER_FEATURES. Rows: 94,029
